# Unsupervised persona discovery
Discovery and causal feature screening for the Persona Vectors limitation. This notebook does not produce confirmatory persona claims.
Protocol: https://github.com/RaphaelKhalid/afterlight/blob/main/research/persona-discovery/PROTOCOL.md
Requires two T4 GPUs. No provider API keys or paid model API calls.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.56.2', 'accelerate==1.10.1', 'datasets==4.1.1', 'huggingface-hub==0.34.4'])


In [ ]:
from pathlib import Path
Path("discovery-contract.json").write_text("{\n  \"studyId\": \"persona-discovery-v1\",\n  \"questionId\": \"q-unsupervised-persona\",\n  \"phase\": \"discovery-and-development-screen\",\n  \"model\": \"Qwen/Qwen2.5-7B-Instruct\",\n  \"modelRevision\": \"a09a35458c702b33eeacc393d103063234e8bc28\",\n  \"sae\": \"andyrdt/saes-qwen2.5-7b-instruct\",\n  \"saeRevision\": \"c37e53c4bb07127ad17ab88f28b93d4e87142e59\",\n  \"saeFolder\": \"resid_post_layer_19/trainer_1\",\n  \"layer\": 19,\n  \"dataset\": \"HuggingFaceH4/ultrachat_200k\",\n  \"datasetRevision\": \"8049631c405ae6576f93f445c6b8166f76f5505a\",\n  \"datasetSplit\": \"train_sft\",\n  \"seed\": 20260914,\n  \"discoveryN\": 1024,\n  \"screenFeatures\": 32,\n  \"screenScenarios\": 12,\n  \"batchSize\": 4,\n  \"maxInputTokens\": 384,\n  \"discoveryMaxNewTokens\": 192,\n  \"screenMaxNewTokens\": 128,\n  \"temperature\": 0.7,\n  \"topP\": 0.95,\n  \"saeTokenStride\": 4,\n  \"featureMinOccurrence\": 0.05,\n  \"featureMaxOccurrence\": 0.8,\n  \"maxDecoderCosine\": 0.8,\n  \"screenNormFraction\": 0.1,\n  \"maxRuntimeSeconds\": 6600,\n  \"apiSpendCapUsd\": 0,\n  \"confirmationStatus\": \"blocked-until-candidates-and-rubrics-frozen\",\n  \"confirmationMaximumCandidates\": 3,\n  \"confirmationScenariosPerCandidate\": 600,\n  \"confirmationConditions\": [\"baseline\", \"sae-positive\", \"sae-negative\", \"optimized-prompt\", \"prompt-extracted-vector\", \"matched-random-direction\"],\n  \"confirmationRepeats\": 2,\n  \"confirmationMaximumResponses\": 21600\n}\r\n", encoding="utf-8")
Path("kaggle_discovery.py").write_text("\"\"\"Kaggle discovery phase. No provider credentials or paid API calls.\"\"\"\nimport os\nos.environ['HF_HOME'] = '/kaggle/temp/persona-hf'\nos.environ['TOKENIZERS_PARALLELISM'] = 'false'\nimport hashlib, json, time, platform, traceback, sys, math\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom datasets import load_dataset\nfrom huggingface_hub import hf_hub_download\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nROOT = Path('/kaggle/working/persona-discovery')\nROOT.mkdir(parents=True, exist_ok=True)\nC = json.loads(Path('discovery-contract.json').read_text(encoding='utf-8-sig'))\nSTART = time.monotonic()\nCONTRACT_HASH = hashlib.sha256(json.dumps(C, sort_keys=True, separators=(',', ':')).encode()).hexdigest()\n\ndef save(name, value):\n    p = ROOT / name\n    temp = p.with_suffix(p.suffix + '.tmp')\n    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')\n    temp.replace(p)\n\ndef event(status, phase, completed, total, message):\n    row = dict(status=status, phase=phase, completed=completed, total=total,\n               message=message, updatedAt=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),\n               contractHash=CONTRACT_HASH, elapsedSeconds=round(time.monotonic()-START, 2))\n    save('status.json', row)\n    with (ROOT/'events.jsonl').open('a', encoding='utf-8') as f:\n        f.write(json.dumps(row)+'\\n')\n    print(json.dumps(row), flush=True)\n\ndef check_time():\n    if time.monotonic()-START > C['maxRuntimeSeconds']:\n        raise TimeoutError('Internal resource limit reached; preserve incomplete phase outputs.')\n\ndef append(name, rows):\n    with (ROOT/name).open('a', encoding='utf-8') as f:\n        for row in rows: f.write(json.dumps(row, ensure_ascii=False)+'\\n')\n\ndef file_hash(path):\n    h=hashlib.sha256()\n    with open(path,'rb') as f:\n        for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)\n    return h.hexdigest()\n\ndef main():\n    save('contract.json', C)\n    assert torch.cuda.device_count() >= 2, 'This frozen runner requires two GPUs; do not silently change precision or model.'\n    event('running','loading',0,C['discoveryN'],'Loading pinned model and SAE; no results yet.')\n    import transformers, datasets, huggingface_hub\n    save('environment.json', dict(python=sys.version,platform=platform.platform(),torch=torch.__version__,\n        transformers=transformers.__version__,datasets=datasets.__version__,huggingface_hub=huggingface_hub.__version__,\n        gpu=[dict(name=torch.cuda.get_device_name(i),memory=torch.cuda.get_device_properties(i).total_memory) for i in range(torch.cuda.device_count())]))\n    tokenizer=AutoTokenizer.from_pretrained(C['model'],revision=C['modelRevision'],padding_side='left')\n    tokenizer.pad_token=tokenizer.eos_token\n    model=AutoModelForCausalLM.from_pretrained(C['model'],revision=C['modelRevision'],torch_dtype=torch.float16,\n        device_map='balanced',max_memory={0:'11GiB',1:'11GiB'},attn_implementation='sdpa').eval()\n    assert not any(str(v) in ('cpu','disk') for v in model.hf_device_map.values()), 'Unexpected CPU offload.'\n    layer=model.model.layers[C['layer']]\n    sae_device=next(layer.parameters()).device\n    checkpoint=hf_hub_download(C['sae'],C['saeFolder']+'/ae.pt',revision=C['saeRevision'])\n    config_path=hf_hub_download(C['sae'],C['saeFolder']+'/config.json',revision=C['saeRevision'])\n    save('sae-config.json',json.loads(Path(config_path).read_text()))\n    state=torch.load(checkpoint,map_location='cpu',weights_only=True)\n    assert tuple(state['encoder.weight'].shape)==(131072,3584)\n    assert tuple(state['decoder.weight'].shape)==(3584,131072)\n    assert int(state['k'])==64 and float(state['threshold'])>=0\n    sae={k:v.to(sae_device,torch.float16) for k,v in state.items() if k in ('encoder.weight','encoder.bias','decoder.weight','b_dec','threshold')}\n    del state\n    save('provenance.json',dict(contractHash=CONTRACT_HASH,saeSha256=file_hash(checkpoint),saeDevice=str(sae_device),\n        deviceMap={k:str(v) for k,v in model.hf_device_map.items()},modelRevision=C['modelRevision'],saeRevision=C['saeRevision'],datasetRevision=C['datasetRevision'],\n        activationConvention='decoder layer index 19 output; generated assistant token positions; every fourth token; threshold inference',precision='float16 model and SAE'))\n    source=load_dataset(C['dataset'],revision=C['datasetRevision'],split=C['datasetSplit'],streaming=True).shuffle(seed=C['seed'],buffer_size=10000)\n    cases=[]; seen=set()\n    for record in source:\n        check_time()\n        messages=record.get('messages',[])\n        if not messages or messages[0].get('role')!='user': continue\n        prompt=messages[0]['content'].strip()\n        key=hashlib.sha256(prompt.encode()).hexdigest()\n        if key in seen or not 20 <= len(prompt) <= 2400: continue\n        text=tokenizer.apply_chat_template([{'role':'system','content':'You are a helpful assistant.'},{'role':'user','content':prompt}],tokenize=False,add_generation_prompt=True)\n        if len(tokenizer.encode(text,add_special_tokens=False)) > C['maxInputTokens']: continue\n        seen.add(key)\n        cases.append(dict(id=key,prompt=prompt,sourceId=record.get('prompt_id'),text=text,\n            split='discovery' if len(cases)<C['discoveryN'] else 'development-screen'))\n        if len(cases)==C['discoveryN']+C['screenScenarios']: break\n    assert len(cases)==C['discoveryN']+C['screenScenarios'], 'Insufficient eligible cases.'\n    save('cases.json',cases)\n    input_device=model.get_input_embeddings().weight.device\n    feature_means=np.lib.format.open_memmap(ROOT/'feature-means.npy',mode='w+',dtype=np.float32,shape=(C['discoveryN'],131072))\n    feature_means[:]=np.nan\n    norms=[]; diagnostics=[]\n\n    def generate(batch,seed,max_new,direction=None):\n        check_time()\n        torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)\n        inputs=tokenizer([x['text'] for x in batch],padding=True,return_tensors='pt',add_special_tokens=False).to(input_device)\n        hook=None\n        if direction is not None:\n            def intervene(_module,_args,output):\n                h=output[0] if isinstance(output,tuple) else output\n                shifted=h+direction.to(device=h.device,dtype=h.dtype)\n                return (shifted,)+output[1:] if isinstance(output,tuple) else shifted\n            hook=layer.register_forward_hook(intervene)\n        try:\n            with torch.inference_mode():\n                ids=model.generate(**inputs,max_new_tokens=max_new,do_sample=True,temperature=C['temperature'],top_p=C['topP'],\n                    pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)\n        finally:\n            if hook is not None: hook.remove()\n        prompt_width=inputs.input_ids.shape[1]\n        rows=[];lens=[]\n        for case,tokens in zip(batch,ids[:,prompt_width:]):\n            values=tokens.tolist()\n            end=next((i+1 for i,t in enumerate(values) if t==tokenizer.eos_token_id),len(values))\n            lens.append(end)\n            rows.append(dict(caseId=case['id'],prompt=case['prompt'],output=tokenizer.decode(values[:end],skip_special_tokens=True),\n                generatedTokenIds=values[:end],generatedTokens=end,truncated=(end==max_new and values[end-1]!=tokenizer.eos_token_id),\n                seed=seed,seedScope='batch',maxNewTokens=max_new,temperature=C['temperature'],topP=C['topP']))\n        return inputs,ids,prompt_width,lens,rows\n\n    def analyze(inputs,ids,width,lens):\n        captured=[]\n        handle=layer.register_forward_hook(lambda _m,_i,o: captured.append((o[0] if isinstance(o,tuple) else o).detach()))\n        mask=torch.cat([inputs.attention_mask,torch.zeros((len(lens),ids.shape[1]-width),device=input_device,dtype=inputs.attention_mask.dtype)],dim=1)\n        for i,n in enumerate(lens): mask[i,width:width+n]=1\n        positions=(mask.cumsum(-1)-1).clamp(min=0)\n        try:\n            with torch.inference_mode(): model(ids,attention_mask=mask,position_ids=positions,use_cache=False)\n        finally: handle.remove()\n        outputs=[]\n        for i,n in enumerate(lens):\n            h=captured[0][i,width:width+n:C['saeTokenStride']]\n            hn=h.float().norm(dim=-1); valid=hn <= 10*hn.median()\n            h=h[valid];hn=hn[valid]\n            assert h.numel()>0 and torch.isfinite(h).all(), 'Invalid sampled activations.'\n            with torch.inference_mode():\n                f=F.relu(F.linear(h-sae['b_dec'],sae['encoder.weight'],sae['encoder.bias']))\n                f=f*(f>sae['threshold'])\n                recon=F.linear(f,sae['decoder.weight'])+sae['b_dec']\n                assert torch.isfinite(f).all() and torch.isfinite(recon).all(), 'Non-finite SAE output.'\n                mean=f.float().mean(0).cpu().numpy()\n                l0=float((f>0).sum(-1).float().mean())\n                error=float(((h.float()-recon.float()).square().sum(-1)/(h.float().square().sum(-1)+1e-8)).mean())\n            outputs.append((mean,hn.cpu().tolist(),dict(sampledTokens=int(h.shape[0]),outliersExcluded=int((~valid).sum()),meanActiveFeatures=l0,relativeSquaredReconstructionError=error)))\n        return outputs\n\n    for start in range(0,C['discoveryN'],C['batchSize']):\n        batch=cases[start:start+C['batchSize']]\n        inputs,ids,width,lens,rows=generate(batch,C['seed']+start,C['discoveryMaxNewTokens'])\n        values=analyze(inputs,ids,width,lens)\n        for offset,(mean,ns,diag) in enumerate(values):\n            feature_means[start+offset]=mean;norms.extend(ns)\n            diag['caseId']=batch[offset]['id'];diagnostics.append(diag)\n        feature_means.flush()\n        append('discovery-responses.jsonl',rows);append('activation-diagnostics.jsonl',[x[2] for x in values])\n        event('running','discovery',start+len(batch),C['discoveryN'],'Collecting responses and SAE activations; no confirmed persona finding.')\n    assert np.isfinite(feature_means).all()\n    mean=np.mean(feature_means,axis=0,dtype=np.float64)\n    variance=np.var(feature_means,axis=0,dtype=np.float64)\n    occurrence=np.mean(feature_means>0,axis=0)\n    scores=variance/(mean**2+1e-8)\n    eligible=(occurrence>=C['featureMinOccurrence'])&(occurrence<=C['featureMaxOccurrence'])&np.isfinite(scores)\n    np.savez_compressed(ROOT/'all-feature-statistics.npz',mean=mean,variance=variance,occurrence=occurrence,score=scores,eligible=eligible)\n    order=sorted(np.where(eligible)[0],key=lambda i:(-scores[i],int(i)))\n    chosen=[];vectors=[];decisions=[]\n    for idx in order:\n        vector=sae['decoder.weight'][:,int(idx)].float();vector=vector/vector.norm()\n        similarity=max([abs(float(torch.dot(vector,v))) for v in vectors],default=0)\n        if similarity>C['maxDecoderCosine']:\n            decisions.append(dict(feature=int(idx),decision='rejected-correlated',cosine=similarity));continue\n        chosen.append(int(idx));vectors.append(vector)\n        decisions.append(dict(feature=int(idx),decision='selected',score=float(scores[idx]),occurrence=float(occurrence[idx])))\n        if len(chosen)==C['screenFeatures']:break\n    save('selection.json',dict(features=chosen,decisions=decisions,eligibleCount=int(eligible.sum()),\n         method='prompt-level coefficient of variation squared; occurrence filter; decoder cosine diversity',traitLabelsUsed=False))\n    if not chosen: raise ValueError('No eligible features. Report negative discovery outcome; no fallback selection.')\n    median_norm=float(np.median(norms));amplitude=median_norm*C['screenNormFraction']\n    save('screen-configuration.json',dict(features=chosen,amplitude=amplitude,medianResidualNorm=median_norm,\n          conditions=['baseline','sae-positive','sae-negative'],developmentOnly=True))\n    for idx in chosen:\n        top=np.argsort(feature_means[:,idx])[-10:][::-1]\n        save('feature-'+str(idx)+'-examples.json',[dict(caseId=cases[int(i)]['id'],meanActivation=float(feature_means[i,idx])) for i in top])\n    dev=cases[C['discoveryN']:]\n    total=len(dev)*(1+2*len(chosen));completed=0\n    conditions=[('baseline',None,None)]+[(sign,idx,v*amplitude*mult) for idx,v in zip(chosen,vectors) for sign,mult in [('sae-positive',1),('sae-negative',-1)]]\n    for condition,feature,direction in conditions:\n        for start in range(0,len(dev),C['batchSize']):\n            _,_,_,_,rows=generate(dev[start:start+C['batchSize']],C['seed']+100000+start,C['screenMaxNewTokens'],direction)\n            for row in rows:row.update(condition=condition,feature=feature,amplitude=0 if direction is None else amplitude,phase='development-screen')\n            append('screen-responses.jsonl',rows);completed+=len(rows)\n            event('running','development-screen',completed,total,'Causal feature screen; candidate interpretation and confirmation remain pending.')\n    event('completed','discovery-and-development-screen',completed,total,'Discovery phase complete. No confirmation results; freeze candidate rubrics and baselines next.')\n    save('artifact-hashes.json',{p.name:file_hash(p) for p in ROOT.iterdir() if p.is_file() and p.name!='artifact-hashes.json'})\n\nif __name__=='__main__':\n    try: main()\n    except Exception as e:\n        previous=json.loads((ROOT/'status.json').read_text()) if (ROOT/'status.json').exists() else {}\n        event('incomplete' if isinstance(e,TimeoutError) else 'failed',previous.get('phase','initialization'),previous.get('completed',0),previous.get('total',C['discoveryN']),str(e))\n        (ROOT/'error.txt').write_text(traceback.format_exc(),encoding='utf-8')\n        raise\r\n", encoding="utf-8")


In [ ]:
subprocess.check_call([sys.executable, 'kaggle_discovery.py'])
